In [5]:

import os
import geopandas as gpd
import requests
import pandas as pd
import folium
import matplotlib.pyplot as plt



In [10]:
urls = {
    "Demografi": "https://geoserver.mapid.io/layers_new/get_layer?api_key=541bb676f4ac4405a003c5fe4476d214&layer_id=68aea69ab2fcc5561a31dda0&project_id=68aea5dbef2411f2c7b64d88&limit=1000", 
    "Emisi Karbon": "https://geoserver.mapid.io/layers_new/get_layer?api_key=541bb676f4ac4405a003c5fe4476d214&layer_id=68aea799b2fcc5561a31dfed&project_id=68aea5dbef2411f2c7b64d88&limit=1000",
    "Industri": "https://geoserver.mapid.io/layers_new/get_layer?api_key=541bb676f4ac4405a003c5fe4476d214&layer_id=68aeaef6b2fcc5561a31f679&project_id=68aea5dbef2411f2c7b64d88&limit=1000" 
   }


#menambahkan kolom skor

def get_data(url, layer_name):
    response = requests.get(url)
    if response.status_code == 200:
        geojson_data = response.json()
        gdf = gpd.GeoDataFrame.from_features(geojson_data['features'])

        #menambahkan kolom skor berdasarkan variabel data
        if layer_name == "Demografi":
            gdf["SKOR_DEMOGRAFI"] = gdf.get("JUMLAH PENDUDUK", "").str.upper().map({"RENDAH": 1, "SEDANG": 2, "TINGGI": 3}).fillna(0)
        elif layer_name == "Emisi Karbon":
            gdf["SKOR_EMISI"] = gdf.get("KELAS", "").str.upper().map({"RENDAH": 1, "SEDANG": 2, "TINGGI": 3}).fillna(0)
        elif layer_name == "Industri":
            gdf["SKOR_DENSITAS"] = gdf.get("DENSITAS", "").str.upper().map({"RENDAH": 1, "SEDANG": 2, "TINGGI": 3}).fillna(0)
        

            return gdf
        return gpd.GeoDataFrame()
    
    #mengambil data dari semua layer
    gdf_demografi = get_data(urls["Demografi"], "Demografi")
    gdf_emisi = get_data(urls["Emisi Karbon"], "Emisi Karbon")
    gdf_densitas = get_data(urls["Industri"], "Industri")
   

    #visualisasi data variabel menggunakan matplotlib
    def visualize_data (gdfs):
        fig, axes = plt.subplots(1, len(gdfs), figsize= (18, 6))
        color_map = ['red', 'blue', 'green']
        for ax, (name, gdf), color in zip (axes, gdfs.items(), color_map):
            gdf.plot(ax=ax, color=color, edgecolor='black')
            ax.set_title(name)
            ax.set_axis_off()
        plt.tight_layout()
        plt.show()
        
        #menampilkan visual dari ketiga variabel/geodataframe
        visualize_data({"Demografi": gdf_demografi, "Emisi Karbon": gdf_emisi, "Industri": gdf_densitas })

       


In [11]:
#mengambil data dari semua layer
gdf_demografi = get_data(urls["Demografi"], "Demografi")
gdf_emisi = get_data(urls["Emisi Karbon"], "Emisi Karbon")
gdf_densitas = get_data(urls["Industri"], "Industri")


#menghitung nilai null
gdf_demografi['SKOR_DEMOGRAFI'] = gdf_demografi ['SKOR_DEMOGRAFI'].fillna(0)
gdf_densitas ['SKOR_DENSITAS'] = gdf_densitas ['SKOR_DENSITAS'].fillna(0)
gdf_emisi ['SKOR_EMISI'] = gdf_emisi['SKOR_BANJIR'].fillna(0)

#analisis intersect / overlay intersect
intersection_gdf = gdf_demografi
if not gdf_densitas.empty:
    intersection_gdf = gpd.overlay(intersection_gdf, gdf_densitas, how= 'intersection', keep_geom_type=False)
if not gdf_emisi.empty:
    intersection_gdf = gpd.overlay(intersection_gdf, gdf_emisi, how='intersection', keep_geom_type=False)

#fix Geometry
intersection_gdf = intersection_gdf[intersection_gdf.is_valid]
intersection_gdf['geometry'] = intersection_gdf['geometry'].apply(lambda x: x.make_valid() if not x.is_valid else x)


#mengatasi nilai null pasca fix geometry
intersection_gdf['SKOR_DEMOGRAFI'] = intersection_gdf['SKOR_DEMOGRAFI'].fillna(0)
intersection_gdf['SKOR_DENSITAS'] = intersection_gdf['SKOR_DENSITAS'].fillna(0)
intersection_gdf['SKOR_EMISI'] = intersection_gdf['SKOR_EMISI'].fillna(0)

#penjumlahan kolom skor di masing masing variabel
intersection_gdf ['SKOR_TOTAL'] = intersection_gdf['SKOR_DEMOGRAFI'] + intersection_gdf['SKOR_DENSITAS'] + intersection_gdf ['SKOR_EMISI']


#klasifikasi
intersection_gdf['KESESUAIAN'] = pd.cut(
    intersection_gdf['SKOR_TOTAL'],
    bins = [2, 5, 7, 9],
    labels = ['RENDAH', 'SEDANG', 'TINGGI'],
    include_lowest =True

)


print(intersection_gdf[['SKOR_DEMOGRAFI', 'SKOR_DENSITAS', 'SKOR_EMISI', 'SKOR_TOTAL', 'KESESUAIAN']].head(10))


AttributeError: Can only use .str accessor with string values!

In [13]:
#simpan ke geojson
import os

output_dir = r"C:\Users\Indra\Downloads\Mapid Academy\Sesi5"
os.makedirs(output_dir, exist_ok=True)

#ambil data multipolygon dan polygon
intersection_gdf_polygon = intersection_gdf[intersection_gdf.geometry.type.isin(['polygon', 'MultiPolygon'])]

#memperbaiki geometry
intersection_gdf_polygon['geometry'] = intersection_gdf_polygon['geometry'].apply(lambda x: x.make_valid()if not x.is_valid else x)

#simpan data geojson
intersection_gdf_polygon.to_file(os.path.join(output_dir, "Hasil_Intersect.json"), driver='GeoJSON')

print (f"Data Berhasil DIsimpan di: {output_dir}")

NameError: name 'intersection_gdf' is not defined